# SVM y MPL_Clasifier

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, accuracy_score,
    precision_recall_fscore_support
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

plt.rcParams.update({'figure.dpi': 120})

# --- Ruta fija para evitar "archivo no encontrado" ---
DATASET_PATH = "/content/drive/My Drive/DATASCIENCE_AI/archive/Base.csv"
print("Usando dataset en:", DATASET_PATH)


Usando dataset en: /content/drive/My Drive/DATASCIENCE_AI/archive/Base.csv


In [3]:
def load_and_preprocess_data(filepath):
    # Carga de datos
    df = pd.read_csv(filepath)

    # Manejo de valores centinela (-1)
    df['prev_address_unknown'] = df['prev_address_months_count'].apply(lambda x: 1 if x == -1 else 0)
    df['prev_address_months_count'] = df['prev_address_months_count'].replace(-1, 0)
    df['bank_months_unknown'] = df['bank_months_count'].apply(lambda x: 1 if x == -1 else 0)
    df['bank_months_count'] = df['bank_months_count'].replace(-1, 0)
    median_session = df[df['session_length_in_minutes'] != -1]['session_length_in_minutes'].median()
    df['session_length_in_minutes'] = df['session_length_in_minutes'].replace(-1, median_session)

    # Eliminación de columnas correlacionadas
    df = df.drop(columns=['velocity_6h', 'velocity_24h'])

    return df

In [4]:
df = load_and_preprocess_data(DATASET_PATH)
# 1. Separar características (X) y variable objetivo (y)
X = df.drop('fraud_bool', axis=1)
y = df['fraud_bool']

# 2. Identificar columnas numéricas y categóricas que irán al pipeline
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# 3. Dividir los datos ANTES de aplicar el pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [5]:
# Función personalizada para el recorte de outliers con IQR
def clip_outliers_iqr(df):
    df_copy = df.copy()
    for col in df_copy.columns:
        Q1 = df_copy[col].quantile(0.25)
        Q3 = df_copy[col].quantile(0.75)
        IQR = Q3 - Q1
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        df_copy[col] = df_copy[col].clip(lower=limite_inferior, upper=limite_superior)
    return df_copy

# 4. Crear el pipeline para transformaciones numéricas
#    Paso 1: Recortar outliers.
#    Paso 2: Escalar los datos.
numeric_transformer = Pipeline(steps=[
    ('outlier_clipper', FunctionTransformer(clip_outliers_iqr)),
    ('scaler', StandardScaler())
])

# 5. Crear el pipeline para transformaciones categóricas
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 6. Combinar los pipelines con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Mantiene columnas que no se especificaron (si las hubiera)
)

pipeline_svc = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', SMOTE(random_state=42)),
    ('classifier', SVC(kernel='linear', random_state=42)) # <-- ÚNICO CAMBIO
])

In [ ]:
print("Entrenando el pipeline con SVC...")
pipeline_svc.fit(X_train, y_train)
print("Entrenamiento completado.")
y_pred_svc = pipeline_svc.predict(X_test)

print("\n--- Reporte de Clasificación (SVC) ---")
print(classification_report(y_test, y_pred_svc))

Entrenando el pipeline con SVC...
